In [1]:
import os

os.environ.setdefault("HIP_VISIBLE_DEVICES", "0") # Force GPU usage instead of iGPU (for ROCm configured torch)

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
import gc

In [2]:
train_df = pd.read_parquet("data/df_train_preprocessed_cutoff44.parquet")
test_df = pd.read_parquet("data/df_test_preprocessed_cutoff44.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

# Guarantee chronological row order before dropping WEEK_NUM bcs TimeSeriesSplit has this as the assumption
train_df = train_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)
test_df = test_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)

# Keep weeknum for our stratisfied sampling during tuning
week_num_train = train_df["WEEK_NUM"].copy()

train_df.drop(columns=id_cols, inplace=True)
test_df.drop(columns=id_cols, inplace=True)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (841500, 192), Test shape: (685159, 192)


## 3.0 Setup

In [3]:
# Define features and target variable
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

In [4]:
# ── Shared tuning setup (used by all three models) ──────────────────────────
# NOTE: TimeSeriesSplit relies on row order. X_train / y_train are assumed to be
# already sorted chronologically (by decision time / WEEK_NUM) upstream, so the
# folds below respect the temporal ordering of the training block.

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_SPLITS = 5      # TimeSeriesSplit folds used during tuning
# Optuna trials per model (raise/lower for your compute budget)
N_TRIALS = 20

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# ── Fixed class weight ──────────────────────────────────────────────────────
# Computed ONCE on the FULL training block and shared, unchanged, across all
# three models. It is NOT recomputed per fold or per trial.
#
# Documented design choice (not a bug): because weight_ratio comes from the full
# training block, the earliest TimeSeriesSplit folds during tuning are scored
# against a class ratio that is partly informed by *later* training data. We
# accept this deliberately — a single fixed, shared weight keeps the three
# models directly comparable and matches the frozen-model evaluation design.
n_positive = int((y_train == 1).sum())
n_negative = int((y_train == 0).sum())
weight_ratio = n_negative / n_positive
print(f"weight_ratio (n_negative / n_positive) = {weight_ratio:.4f}")


def gini(y_true, y_score):
    """Gini coefficient = 2 * AUC - 1"""
    return 2.0 * roc_auc_score(y_true, y_score) - 1.0


def print_trial(study, trial):
    """Optuna callback: print the Gini and params of each completed trial."""
    print(
        f"  Trial {trial.number:>3} | Gini: {trial.value:.4f} | Params: {trial.params}")


def make_tuning_subsample(X, y, week_num, frac, seed=SEED):
    """Optional stratified subsample of the training block, for TUNING ONLY.

    Stratifies jointly on (WEEK_NUM, target): groups rows by week and class,
    draws frac of the rows within each group, then restores the original row
    order (by position). Returns the data unchanged when frac is None or >= 1.
    """
    if frac is None or frac >= 1.0:
        return X, y
    rng = np.random.default_rng(seed)
    y_arr = y.to_numpy()
    week_arr = week_num.to_numpy()

    keep_parts = []
    for week in np.unique(week_arr):
        week_mask = week_arr == week
        for cls in np.unique(y_arr):
            idx = np.where(week_mask & (y_arr == cls))[0]
            n_keep = max(1, round(len(idx) * frac)) if len(idx) > 0 else 0
            if n_keep > 0:
                keep_parts.append(rng.choice(idx, size=n_keep, replace=False))
    keep = np.sort(np.concatenate(keep_parts))
    return X.iloc[keep], y.iloc[keep]


# Fraction of the training block used during tuning to reduce compute cost for optuna. The final model is trained on the full training block.
FRAC_LR = 0.3
FRAC_XGB = 0.3
FRAC_MLP = 0.3

weight_ratio (n_negative / n_positive) = 32.6076


## 3.1 Logistische regressie

In [5]:
# Optuna tuning — Logistic Regression
X_lr, y_lr = make_tuning_subsample(X_train, y_train, week_num_train, FRAC_LR)

# Convert ONCE to a C-contiguous float64 array. X_lr mixes float64 and int8
# columns, so handing the DataFrame straight to .fit() makes sklearn copy and
# promote the whole block on every single fit (N_TRIALS * N_SPLITS times).
X_lr_np = np.ascontiguousarray(X_lr.to_numpy(dtype=np.float64))
y_lr_np = y_lr.to_numpy()

# Materialise the folds once; they are identical across trials.
LR_FOLDS = list(tscv.split(X_lr_np))

# # Looser tolerance during the SEARCH only: it changes the fold Gini far below
# # the digit we rank trials on, but saves a lot of liblinear iterations. The
# # final model below is refit at sklearn's default tol (1e-4).
LR_TUNE_TOL = 1e-4


def lr_solver(l1_ratio):
    # sklearn 1.8+: penalty is deprecated, l1_ratio drives the penalty.
    # l1_ratio == 0.0 -> pure L2  -> newton-cholesky (2nd order, fast/accurate)
    # l1_ratio == 1.0 -> pure L1  -> liblinear (coordinate descent, purpose-built
    #   for L1; far faster and more robust than saga, which struggles to converge
    #   on pure L1 because there is no L2 term to add strong convexity).
    return "newton-cholesky" if l1_ratio == 0.0 else "liblinear"


def lr_fit_fold(X, y, tr_idx, va_idx, C, l1_ratio, solver, tol):
    """Fit + score one TimeSeriesSplit fold. X/y are passed as arguments (not
    captured as globals) so joblib memmaps them to the workers instead of
    pickling a full copy per fold."""
    model = LogisticRegression(
        C=C,
        l1_ratio=l1_ratio,
        solver=solver,
        class_weight={0: 1.0, 1: weight_ratio},
        max_iter=1000,
        tol=tol,
        random_state=SEED,
    )
    model.fit(X[tr_idx], y[tr_idx])
    proba = model.predict_proba(X[va_idx])[:, 1]
    return gini(y[va_idx], proba)


def lr_objective(trial):
    # Upper bound kept at 1e1: the features are z-scored (see m2, 2.7), so
    # C >= 1e1 is effectively unpenalised — the region where liblinear needs
    # the most coordinate-descent iterations and where L1 stops being sparse.
    # If the selected C ends up pinned at 1e1, widen this again.
    C = trial.suggest_float("C", 1e-4, 1e1, log=True)
    l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 1.0])
    solver = lr_solver(l1_ratio)

    # liblinear is single-threaded, so the 5 folds are run as separate
    # processes. Speedup is bounded by the LARGEST fold (TimeSeriesSplit folds
    # grow), so expect ~3x rather than 5x. The TPE sampler itself stays
    # sequential, which keeps the search reproducible.
    fold_ginis = Parallel(n_jobs=N_SPLITS)(
        delayed(lr_fit_fold)(
            X_lr_np, y_lr_np, tr_idx, va_idx, C, l1_ratio, solver, LR_TUNE_TOL
        )
        for tr_idx, va_idx in LR_FOLDS
    )
    return float(np.mean(fold_ginis))


lr_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
lr_study.optimize(lr_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best LR params:", lr_study.best_params)
print(f"Best LR mean CV Gini: {lr_study.best_value:.4f}")

# Retrain the final LR on the FULL training block with the selected params.
# Mirror the objective's solver routing based on the chosen l1_ratio.
lr_best = LogisticRegression(
    **lr_study.best_params,
    solver=lr_solver(lr_study.best_params["l1_ratio"]),
    class_weight={0: 1.0, 1: weight_ratio},
    max_iter=1000,
    random_state=SEED,
)
lr_best.fit(X_train, y_train)

gc.collect()

  Trial   0 | Gini: 0.6104 | Params: {'C': 0.0074593432857265485, 'l1_ratio': 0.0}
  Trial   1 | Gini: 0.6074 | Params: {'C': 0.09846738873614563, 'l1_ratio': 0.0}
  Trial   2 | Gini: 0.6148 | Params: {'C': 0.00019517224641449495, 'l1_ratio': 0.0}
  Trial   3 | Gini: 0.6047 | Params: {'C': 0.3470266988650412, 'l1_ratio': 1.0}
  Trial   4 | Gini: 0.6036 | Params: {'C': 1.452824663751602, 'l1_ratio': 0.0}
  Trial   5 | Gini: 0.5959 | Params: {'C': 0.0008260808399079611, 'l1_ratio': 1.0}
  Trial   6 | Gini: 0.6117 | Params: {'C': 0.01444525102276306, 'l1_ratio': 1.0}
  Trial   7 | Gini: 0.5847 | Params: {'C': 0.0004982752357076451, 'l1_ratio': 1.0}
  Trial   8 | Gini: 0.6093 | Params: {'C': 0.019069966103000432, 'l1_ratio': 0.0}
  Trial   9 | Gini: 0.6086 | Params: {'C': 0.03725393839578886, 'l1_ratio': 0.0}
  Trial  10 | Gini: 0.6026 | Params: {'C': 5.771258219920906, 'l1_ratio': 0.0}
  Trial  11 | Gini: 0.4624 | Params: {'C': 0.0001090778569000611, 'l1_ratio': 1.0}
  Trial  12 | Gini: 0

80

## 3.2 XGBoost

In [6]:
# Optuna tuning — XGBoost
X_xgb, y_xgb = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_XGB)


def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_xgb):
        model = XGBClassifier(
            **params,
            scale_pos_weight=weight_ratio,  # fixed, shared weight (not tuned)
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        model.fit(X_xgb.iloc[tr_idx], y_xgb.iloc[tr_idx])
        proba = model.predict_proba(X_xgb.iloc[va_idx])[:, 1]
        fold_ginis.append(gini(y_xgb.iloc[va_idx], proba))
    return float(np.mean(fold_ginis))


xgb_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best XGB params:", xgb_study.best_params)
print(f"Best XGB mean CV Gini: {xgb_study.best_value:.4f}")

# Retrain the final XGBoost on the FULL training block with the selected params.
xgb_best = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=weight_ratio,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)
xgb_best.fit(X_train, y_train)

gc.collect()

  Trial   0 | Gini: 0.4731 | Params: {'max_depth': 5, 'learning_rate': 0.22648248189516848, 'n_estimators': 750, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 4, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}
  Trial   1 | Gini: 0.5795 | Params: {'max_depth': 7, 'learning_rate': 0.05675206026988748, 'n_estimators': 100, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 5, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}
  Trial   2 | Gini: 0.6135 | Params: {'max_depth': 5, 'learning_rate': 0.0199473547030745, 'n_estimators': 500, 'subsample': 0.645614570099021, 'colsample_bytree': 0.8059264473611898, 'min_child_weight': 3, 'reg_alpha': 4.258943089524393e-06, 'reg_lambda': 1.9826980964985924e-05}
  Trial   3 | Gini: 0.5469 | Params: {'max_depth': 6, 'learning_rate': 0.08810003129071789, 'n_estimators': 250, 'subsample': 0.7571172192068059, 'colsample

1282

## 3.3 MLP

In [7]:
# Optuna tuning — MLP (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FEATURES = X_train.shape[1]
MLP_EPOCHS = 15  # fixed training budget per fit (kept small for tuning)

# Fixed candidate architectures (hidden-layer sizes) keep the search space small.
MLP_ARCHITECTURES = {
    "128": [128],
    "256-128": [256, 128],
    "128-64": [128, 64],
}


class MLP(nn.Module):
    def __init__(self, n_features, hidden_sizes, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))  # single output logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(X_tr, y_tr, hidden_sizes, lr, weight_decay, batch_size, dropout):
    torch.manual_seed(SEED)
    model = MLP(N_FEATURES, hidden_sizes, dropout).to(device)
    # Fixed, shared class weight applied through the loss.
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(weight_ratio, device=device)
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay)

    ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    # shuffle=False keeps the time order of the (already ordered) training rows.
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.train()
    for _ in range(MLP_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict_mlp(model, X_va):
    model.eval()
    xb = torch.tensor(X_va, dtype=torch.float32).to(device)
    return torch.sigmoid(model(xb)).cpu().numpy()


# MLP tuning runs on a stratified, time-ordered subsample (see FRAC_MLP).
X_mlp, y_mlp = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_MLP)
X_mlp_np = X_mlp.to_numpy(dtype=np.float32)
y_mlp_np = y_mlp.to_numpy(dtype=np.float32)


def mlp_objective(trial):
    arch_key = trial.suggest_categorical(
        "architecture", list(MLP_ARCHITECTURES))
    hidden_sizes = MLP_ARCHITECTURES[arch_key]
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_mlp_np):
        model = train_mlp(
            X_mlp_np[tr_idx], y_mlp_np[tr_idx],
            hidden_sizes, lr, weight_decay, batch_size, dropout,
        )
        proba = predict_mlp(model, X_mlp_np[va_idx])
        fold_ginis.append(gini(y_mlp_np[va_idx], proba))
    return float(np.mean(fold_ginis))


mlp_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
mlp_study.optimize(mlp_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best MLP params:", mlp_study.best_params)
print(f"Best MLP mean CV Gini: {mlp_study.best_value:.4f}")

# Retrain the final MLP on the FULL training block with the selected params.
best = mlp_study.best_params
mlp_best = train_mlp(
    X_train.to_numpy(dtype=np.float32),
    y_train.to_numpy(dtype=np.float32),
    MLP_ARCHITECTURES[best["architecture"]],
    best["learning_rate"],
    best["weight_decay"],
    best["batch_size"],
    best["dropout"],
)

gc.collect()

  Trial   0 | Gini: 0.5000 | Params: {'architecture': '256-128', 'learning_rate': 0.0015751320499779737, 'weight_decay': 4.2079886696066345e-06, 'batch_size': 1024, 'dropout': 0.3005575058716044}
  Trial   1 | Gini: 0.5097 | Params: {'architecture': '128-64', 'learning_rate': 0.004622589001020831, 'weight_decay': 7.068974950624607e-06, 'batch_size': 1024, 'dropout': 0.2623782158161189}
  Trial   2 | Gini: 0.6077 | Params: {'architecture': '128-64', 'learning_rate': 0.00019010245319870352, 'weight_decay': 1.4742753159914662e-05, 'batch_size': 1024, 'dropout': 0.09983689107917987}
  Trial   3 | Gini: 0.5138 | Params: {'architecture': '256-128', 'learning_rate': 0.0016409286730647919, 'weight_decay': 4.809461967501575e-06, 'batch_size': 1024, 'dropout': 0.40419867405823057}
  Trial   4 | Gini: 0.5266 | Params: {'architecture': '128-64', 'learning_rate': 0.0007591104805282694, 'weight_decay': 3.0771802712506896e-06, 'batch_size': 1024, 'dropout': 0.12938999080000846}
  Trial   5 | Gini: 0.

0

## 3.4 Summary + save models

In [8]:
# ── Tuning summary ──────────────────────────────────────────────────────────
print(f"Fixed weight_ratio (shared across all models): {weight_ratio:.4f}")
print(f"Random seed: {SEED}\n")
print("Best hyperparameters per model")
print("  LR :", lr_study.best_params, f"(CV Gini {lr_study.best_value:.4f})")
print("  XGB:", xgb_study.best_params, f"(CV Gini {xgb_study.best_value:.4f})")
print("  MLP:", mlp_study.best_params, f"(CV Gini {mlp_study.best_value:.4f})")

Fixed weight_ratio (shared across all models): 32.6076
Random seed: 42

Best hyperparameters per model
  LR : {'C': 0.00019517224641449495, 'l1_ratio': 0.0} (CV Gini 0.6148)
  XGB: {'max_depth': 5, 'learning_rate': 0.01758290577602986, 'n_estimators': 450, 'subsample': 0.5934770033321395, 'colsample_bytree': 0.7628614131457858, 'min_child_weight': 20, 'reg_alpha': 9.393382777352802e-05, 'reg_lambda': 7.3210985188372955e-06} (CV Gini 0.6222)
  MLP: {'architecture': '128', 'learning_rate': 0.0002810986131418948, 'weight_decay': 0.009639757903159522, 'batch_size': 256, 'dropout': 0.33362280523931753} (CV Gini 0.6248)


In [9]:
import os
import joblib

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Logistic Regression — plain pickle via joblib
lr_path = os.path.join(MODELS_DIR, "lr_best_44.joblib")
joblib.dump(lr_best, lr_path)

# XGBoost — native format (portable across xgboost/sklearn versions)
xgb_path = os.path.join(MODELS_DIR, "xgb_best_44.json")
xgb_best.save_model(xgb_path)

# MLP — TorchScript: architectuur én gewichten in één self-contained archief.
# Voordeel t.o.v. een losse state_dict: om te scoren hoeft de MLP-class niet
# opnieuw gedefinieerd te worden (zie results.ipynb, dat enkel torch.jit.load doet).
mlp_path = os.path.join(MODELS_DIR, "mlp_best_44.pt")
torch.jit.save(torch.jit.script(mlp_best.eval()), mlp_path)

print(f"Saved LR  -> {lr_path}")
print(f"Saved XGB -> {xgb_path}")
print(f"Saved MLP -> {mlp_path}")

Saved LR  -> models/lr_best_44.joblib
Saved XGB -> models/xgb_best_44.json
Saved MLP -> models/mlp_best_44.pt
